## Alignment - MNIST example GEODESICS

Goals:

* Show alignment methods on real learned data
* train vaes and construct manifolds
* plot manifolds with data (three classes)
* of course use geodesics:)

Hypothesis: 
* If latent distributions are non-symmetric AND geometry is non-symmetric, then GW can find a unique minimizer (for all classes iff the classes have inter distinguishable distributions) and in theory a perfect alignment.

In [1]:
import numpy as np
import scipy
import matplotlib.pyplot as plt
import torch
import pandas as pd

from sklearn.decomposition import PCA
from scipy.optimize import minimize
from scipy.stats import vonmises_fisher
from scipy.spatial.distance import cdist, pdist, squareform

from colors import color_segment
from supervised_vae import SupervisedVAE
from train_two_vaes import train_two_vaes
from plot_vaes import plot_vae_latent_alignment
from utils_GW import *
from geodesics import *

blue_pink = color_segment()

Setup:

1. `train_two_vaes.py` splits MNIST dataset once (should happen in training script) into four sets {train, val, align, test}
2. `mnist_geodesics.ipynb` notebook loads saved splits and (trains) models
    * train both VAE A and VAE B on the same train / val split but using different seeds
3. choose 3 anchor images per class from the align split using a random seed
4. compute 36 pairwise (geodesic) distances per domain (A and B) use the linear energy approximation
5. run GW on the two 9x9 distance matrices
    * finding a global map from B to A: using an affine baseline with a fixed parameterization consisting of a rotation and translation
    * finding a global map from B to A: but use a neural network to learn the parametrization of the map
6. validate on unseen test images:
    * apply the map to the latent space B (getting aligned B) and overlaying with latent space A : these should fit (use same PCA mapping)
    * or advanced validate: sample from latent space B (an image neither trained, validated nor used for alignment) and decode in latent space A : the class should be the same for the two + the image should display an unambiguous digit.

In [ ]:
# load vaes from checkpoints instead of training new ones
# Instantiate the models
vaeA = SupervisedVAE(latent_dim=8, num_classes=3)
vaeB = SupervisedVAE(latent_dim=8, num_classes=3)

classes = '3classes'

# Load the state dicts
vaeA.load_state_dict(torch.load(f"checkpoints/mnist_same_{classes}_vaeA.pt"))
vaeB.load_state_dict(torch.load(f"checkpoints/mnist_same_{classes}_vaeB.pt"))

# Load latent variables
zA_dict = torch.load(f"checkpoints/mnist_same_{classes}_zA.pt")
zB_dict = torch.load(f"checkpoints/mnist_same_{classes}_zB.pt")

zA = zA_dict["z"]
zB = zB_dict["z"]
y = zA_dict["y"]  # or zB_dict["y"], since yA and yB should be the same

In [ ]:
# # train vaes
# vaeA, vaeB, zA, zB, y = train_two_vaes(
#     device="cpu",
#     epochs=15,
#     latent_dim=8,
#     prefix="checkpoints/mnist_same_classes",
#     num_classes=3,
#     seedA=0,
#     seedB=42
# )

Training VAE_A...
[Epoch   1] Loss=193.5448 Recon=188.0639 KL=4.5711 CLS=0.9099
[Epoch   2] Loss=134.6009 Recon=125.4307 KL=8.6795 CLS=0.4907
[Epoch   3] Loss=119.4224 Recon=108.2441 KL=10.7939 CLS=0.3843
[Epoch   4] Loss=113.0289 Recon=101.3004 KL=11.4091 CLS=0.3194
[Epoch   5] Loss=109.1963 Recon=97.2170 KL=11.7049 CLS=0.2745
[Epoch   6] Loss=105.4920 Recon=93.0653 KL=12.1911 CLS=0.2356
[Epoch   7] Loss=103.2219 Recon=90.4266 KL=12.5857 CLS=0.2096
[Epoch   8] Loss=101.2209 Recon=88.1373 KL=12.8936 CLS=0.1899
[Epoch   9] Loss=99.6128 Recon=86.2287 KL=13.2070 CLS=0.1771
[Epoch  10] Loss=98.2170 Recon=84.5916 KL=13.4575 CLS=0.1678
[Epoch  11] Loss=97.0673 Recon=83.2100 KL=13.6985 CLS=0.1587
[Epoch  12] Loss=95.8085 Recon=81.7531 KL=13.9082 CLS=0.1472
[Epoch  13] Loss=95.0379 Recon=80.8712 KL=14.0281 CLS=0.1386
[Epoch  14] Loss=94.3473 Recon=80.0946 KL=14.1211 CLS=0.1317
[Epoch  15] Loss=93.6860 Recon=79.3256 KL=14.2350 CLS=0.1254
Training VAE_B...
[Epoch   1] Loss=195.5822 Recon=190.200

In [ ]:
# choose 9 fixed point samples common to both domains
num_points_per_class = 3
fixed_points_A = []

